# Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import cv2
import torch
import ast
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from pathlib import Path

from ultralytics import YOLO
from torchvision.ops import box_convert
from boxmot import DeepOcSort

In [ ]:
ROOT_DIR  = 'tensorflow-great-barrier-reef/'
CKPT_PATH = 'output/yolov8/gbr_yolov8n_fold/weights/best.pt'
IMG_SIZE  = 4800
CONF      = 0.25 
IOU       = 0.7
device    = '0' if torch.cuda.is_available() else 'cpu'

In [ ]:
def get_path(row):
    row['image_path'] = f'{ROOT_DIR}/train_images/video_{row.video_id}/{row.video_frame}.jpg'
    return row

df = pd.read_csv(f'{ROOT_DIR}/train.csv')
tqdm.pandas()
df = df.progress_apply(get_path, axis=1)
df['annotations'] = df['annotations'].progress_apply(lambda x: ast.literal_eval(x))
df['num_bbox'] = df['annotations'].progress_apply(lambda x: len(x))

# Helper

In [ ]:
def convert_boxes(boxes, in_fmt, out_fmt):
    if len(boxes) == 0:
        return boxes
    
    if not isinstance(boxes, torch.Tensor):
        boxes = torch.tensor(boxes, dtype=torch.float32)
        
    converted = box_convert(boxes, in_fmt=in_fmt, out_fmt=out_fmt)
    return converted.numpy()

In [ ]:
def draw_bboxes(img, bboxes, confs=None, track_ids=None, color=(0, 255, 0)):
    image = img.copy()
    
    for idx, box in enumerate(bboxes):
        x1, y1, x2, y2 = map(int, box)
        
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2, cv2.LINE_AA)
        
        label = "Starfish"
        if track_ids is not None:
            label += f" ID:{int(track_ids[idx])}"
        if confs is not None:
            label += f" {confs[idx]:.2f}"
            
        t_size = cv2.getTextSize(label, 0, fontScale=0.5, thickness=1)[0]
        c2 = x1 + t_size[0], y1 - t_size[1] - 3
        cv2.rectangle(image, (x1, y1), c2, color, -1, cv2.LINE_AA)
        cv2.putText(image, label, (x1, y1 - 2), 0, 0.5, [255, 255, 255], thickness=1, lineType=cv2.LINE_AA)
        
    return image

In [ ]:
print(f"Loading YOLOv8 model from {CKPT_PATH}...")
model = YOLO(CKPT_PATH) 

tracker = DeepOcSort(
    reid_weights=Path('osnet_x0_25_msmt17.pt'),
    device='0',
    half=True,
)

# Inference

## Run Inference on **Train**

In [ ]:
video_candidates = df[df.num_bbox > 0].groupby('video_id').size().sort_values(ascending=False).head(5)
print("Videos with most annotations:")
print(video_candidates)

selected_video_id = video_candidates.index[0]
print(f"\nSelected video_id: {selected_video_id}")

video_df = df[df.video_id == selected_video_id].sort_values('video_frame').reset_index(drop=True)
print(f"Total frames in video: {len(video_df)}")
print(f"Frames with bbox: {(video_df.num_bbox > 0).sum()}")

tracker = DeepOcSort(
    reid_weights=Path('osnet_x0_25_msmt17.pt'),
    device='0',
    half=True,
)

image_paths = video_df.head(30).image_path.tolist() 

for idx, path in enumerate(tqdm(image_paths, desc="Processing frames")):
    img = cv2.imread(path)
    if img is None:
        print(f"Warning: Cannot read {path}")
        continue
        
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    results = model.predict(
        img_rgb, 
        conf=CONF, 
        iou=IOU, 
        imgsz=IMG_SIZE,
        device=device,
        verbose=False
    )[0]
    
    dets = results.boxes.data.cpu().numpy()
    
    final_boxes, track_ids, confs = [], [], []
    
    if len(dets) > 0:
        tracks = tracker.update(dets, img_rgb)
        if len(tracks) > 0:
            final_boxes = tracks[:, :4]
            track_ids = tracks[:, 4]
            confs = tracks[:, 5]
    
    if len(final_boxes) > 0:
        annotated_img = draw_bboxes(img_rgb, final_boxes, confs, track_ids)
    else:
        annotated_img = img_rgb

    if idx % 5 == 0 or len(final_boxes) > 0:
        plt.figure(figsize=(15, 8))
        plt.imshow(annotated_img)
        plt.axis('off')
        plt.title(f"Frame {idx} (video_frame: {video_df.iloc[idx]['video_frame']}) - Detected: {len(final_boxes)} objects")
        plt.show()